# Oriented Ships

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import numpy as np
import math
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True

In [ ]:
class OrientedShipDataset(Dataset):
    def __init__(self, image_dir, label_dir, S=16, img_size=128):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.S = S
        self.img_size = img_size
        self.image_files =[f for f in os.listdir(image_dir) if f.endswith('.jpg')]

        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.cache = [self.transform(Image.open(os.path.join(image_dir, f)).convert("RGB"))
                      for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        label_path = os.path.join(self.label_dir, img_name.replace('.jpg', '.txt'))

        image = self.cache[idx]

        # Target tensor:[S, S, 6] -> Conf, cx_cell, cy_cell, w_norm, h_norm, theta
        target = torch.zeros((self.S, self.S, 6))

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cx, cy, w, h, theta = map(float, parts)

                        grid_x, grid_y = int(cx * self.S), int(cy * self.S)

                        cx_cell = cx * self.S - grid_x
                        cy_cell = cy * self.S - grid_y

                        if grid_x < self.S and grid_y < self.S:
                            target[grid_y, grid_x, :] = torch.tensor([1.0, cx_cell, cy_cell, w, h, theta])

        return image, target

BASE_DIR = r"/kaggle/input/competitions/oriented-ship-aicc-round-7/Oriented"
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'images', 'train')
TRAIN_LBL_DIR = os.path.join(BASE_DIR, 'labels', 'train')

train_dataset = OrientedShipDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR, S=8, img_size=256)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print(f"Loaded {len(train_dataset)} training images.")

In [ ]:
import matplotlib.patches as patches
import matplotlib.transforms as transforms

images, targets = next(iter(train_loader))

img = images[0]
tgt = targets[0]

mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

img = (img * std + mean).clamp(0,1).permute(1,2,0).numpy()

S = 8
H, W = img.shape[:2]

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")

for y in range(S):
    for x in range(S):
        if tgt[y, x, 0] > 0.5:
            cx_c, cy_c, w, h, th = tgt[y, x, 1:].tolist()

            cx = (x + cx_c) / S * W
            cy = (y + cy_c) / S * H

            w *= W
            h *= H

            r = patches.Rectangle(
                (cx - w/2, cy - h/2),
                w, h,
                fill=False,
                edgecolor="lime",
                linewidth=2
            )

            r.set_transform(
                transforms.Affine2D().rotate_around(cx, cy, th) + plt.gca().transData
            )

            plt.gca().add_patch(r)
            plt.plot(cx, cy, "w.", markersize=4)

plt.show()

In [ ]:
class BaselineYOLO(nn.Module):
    def __init__(self, S=8):
        super(BaselineYOLO, self).__init__()
        self.S = S

        resnet = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        self.head = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 5, kernel_size=1)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        out = out.permute(0, 2, 3, 1)

        return self.sigmoid(out)

In [ ]:
class YOLO_HBB_Loss(nn.Module):
    def __init__(self, lambda_coord=5.0, lambda_noobj=0.5):
        super(YOLO_HBB_Loss, self).__init__()
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.mse = nn.MSELoss(reduction='sum')

    def forward(self, predictions, target):
        obj_mask = target[..., 0] == 1
        noobj_mask = target[..., 0] == 0

        noobj_loss = self.mse(predictions[..., 0][noobj_mask], target[..., 0][noobj_mask])

        obj_loss = self.mse(predictions[..., 0][obj_mask], target[..., 0][obj_mask])

        xy_loss = self.mse(predictions[..., 1:3][obj_mask], target[..., 1:3][obj_mask])

        pred_wh = torch.sqrt(predictions[..., 3:5][obj_mask] + 1e-6)
        target_wh = torch.sqrt(target[..., 3:5][obj_mask] + 1e-6)
        wh_loss = self.mse(pred_wh, target_wh)

        total_loss = (self.lambda_coord * xy_loss) + \
                     (self.lambda_coord * wh_loss) + \
                     obj_loss + \
                     (self.lambda_noobj * noobj_loss)

        return total_loss / predictions.size(0)

In [ ]:
model = BaselineYOLO(S=16).to(device)
criterion = YOLO_HBB_Loss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

epochs = 60

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    loop = tqdm(train_loader, leave=True)
    for images, targets in loop:
        images = images.to(device)
        targets = targets.to(device)

        predictions = model(images)

        loss = criterion(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Average Loss: {epoch_loss / len(train_loader):.4f}")

## Submission

In [ ]:
from torchvision import transforms as T
from shapely.geometry import Polygon
from tqdm import tqdm

def get_rotated_polygon(cx, cy, w, h, theta):
    corners = np.array([
        [-w/2, -h/2],
        [ w/2, -h/2],
        [ w/2,  h/2],
        [-w/2,  h/2]
    ])

    c, s = np.cos(theta), np.sin(theta)
    R = np.array([[c, -s],
                  [s,  c]])

    rotated_corners = np.dot(corners, R.T) + np.array([cx, cy])
    return Polygon(rotated_corners)

In [ ]:
def decode_predictions(predictions, S=16, conf_thresh=0.5):
    boxes =[]
    for i in range(S):
        for j in range(S):
            conf = predictions[i, j, 0].item()
            if conf > conf_thresh:
                cx_cell = predictions[i, j, 1].item()
                cy_cell = predictions[i, j, 2].item()
                w = predictions[i, j, 3].item()
                h = predictions[i, j, 4].item()

                cx = (j + cx_cell) / S
                cy = (i + cy_cell) / S
                theta = 0.0

                boxes.append((conf, cx, cy, w, h, theta))
    return boxes


In [ ]:
import pandas as pd

VAL_DIR = r"/kaggle/input/competitions/oriented-ship-aicc-round-7/Oriented"
VAL_IMG_DIR = os.path.join(VAL_DIR, "images", 'val')
VAL_LBL_DIR = os.path.join(VAL_DIR, "labels", 'val')

transform = T.Compose([
      T.Resize((512, 512)),
      T.ToTensor(),
      T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model.eval()

test_images = [f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.jpg')]

submission_data = []

with torch.no_grad():
    for img_name in tqdm(test_images):

        img_path = os.path.join(VAL_IMG_DIR, img_name)

        image = Image.open(img_path).convert("RGB")

        img_tensor = transform(image).unsqueeze(0).to(device)

        output = model(img_tensor).squeeze(0)

        pred_boxes = decode_predictions(output, S=16, conf_thresh=0.5)

        res_strs = []

        for box in pred_boxes:
            conf, cx, cy, w, h, theta = box

            res_strs.append(
                f"{conf:.4f} {cx:.4f} {cy:.4f} {w:.4f} {h:.4f} {theta:.4f}"
            )

        result_string = ", ".join(res_strs)

        if not result_string:
            result_string = "0 0.5 0.5 0.01 0.01 0" # to avoid 'null' error during submission

        submission_data.append({
            "Id": img_name,
            "result": result_string
        })

submission_df = pd.DataFrame(submission_data)

submission_df.to_csv("submission.csv", index=False)

print(submission_df.head())